# Pandas + SQLite3: prosty projekt demonstracyjny

Ten notebook pokazuje pełny, mały przepływ pracy:

1. utworzenie lokalnej bazy SQLite,
2. stworzenie tabel i przykładowych danych,
3. odczyt danych z SQLite do ramek `pandas.DataFrame`,
4. analiza danych w pandas,
5. zapytania SQL z poziomu Pythona,
6. zapis wyników analizy z powrotem do SQLite,
7. eksport wyniku do CSV.

Notebook jest samowystarczalny: nie wymaga żadnego zewnętrznego pliku z danymi. Po uruchomieniu utworzy plik `sklep_demo.sqlite` w tym samym katalogu.


## 1. Import bibliotek

`sqlite3` jest biblioteką standardową Pythona, więc nie trzeba jej instalować.  
`pandas` służy do pracy z ramkami danych.  
`matplotlib` wykorzystamy tylko do prostego wykresu demonstracyjnego.


In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

DB_PATH = Path('sklep_demo.sqlite')
DB_PATH


## 2. Utworzenie bazy SQLite

Dla przejrzystości usuwamy starą wersję bazy, jeżeli istnieje. Dzięki temu notebook można uruchamiać wiele razy i zawsze otrzymać ten sam wynik.


In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('PRAGMA foreign_keys = ON;')

print(f'Utworzono bazę: {DB_PATH.resolve()}')


## 3. Definicja tabel

Tworzymy trzy proste tabele:

- `klienci`,
- `produkty`,
- `sprzedaz`.

Tabela `sprzedaz` ma klucze obce do tabel `klienci` i `produkty`.


In [ ]:
cursor.executescript('''
CREATE TABLE klienci (
    klient_id INTEGER PRIMARY KEY,
    imie TEXT NOT NULL,
    miasto TEXT NOT NULL,
    segment TEXT NOT NULL
);

CREATE TABLE produkty (
    produkt_id INTEGER PRIMARY KEY,
    nazwa TEXT NOT NULL,
    kategoria TEXT NOT NULL,
    cena REAL NOT NULL
);

CREATE TABLE sprzedaz (
    transakcja_id INTEGER PRIMARY KEY,
    data TEXT NOT NULL,
    klient_id INTEGER NOT NULL,
    produkt_id INTEGER NOT NULL,
    ilosc INTEGER NOT NULL,
    rabat REAL NOT NULL,
    FOREIGN KEY (klient_id) REFERENCES klienci(klient_id),
    FOREIGN KEY (produkt_id) REFERENCES produkty(produkt_id)
);
''')

conn.commit()
print('Tabele zostały utworzone.')


## 4. Wstawienie danych demonstracyjnych

Dane są niewielkie, żeby dało się je łatwo omówić podczas szkolenia lub prezentacji.


In [ ]:
klienci = [
    (1, 'Anna', 'Lublin', 'B2C'),
    (2, 'Piotr', 'Warszawa', 'B2B'),
    (3, 'Maria', 'Kraków', 'B2C'),
    (4, 'Tomasz', 'Gdańsk', 'B2B'),
    (5, 'Ewa', 'Poznań', 'B2C'),
    (6, 'Kamil', 'Wrocław', 'B2B'),
]

produkty = [
    (1, 'Herbata zielona', 'Napoje', 18.50),
    (2, 'Kawa arabica', 'Napoje', 32.90),
    (3, 'Czekolada gorzka', 'Słodycze', 12.40),
    (4, 'Miód lipowy', 'Delikatesy', 28.00),
    (5, 'Makaron pełnoziarnisty', 'Produkty suche', 8.90),
]

sprzedaz = [
    (1, '2026-01-05', 1, 1, 3, 0.00),
    (2, '2026-01-08', 2, 2, 5, 0.05),
    (3, '2026-01-12', 3, 3, 2, 0.00),
    (4, '2026-02-03', 1, 4, 1, 0.10),
    (5, '2026-02-15', 4, 1, 8, 0.08),
    (6, '2026-02-21', 5, 5, 4, 0.00),
    (7, '2026-03-02', 6, 2, 6, 0.12),
    (8, '2026-03-09', 2, 1, 10, 0.15),
    (9, '2026-03-19', 3, 4, 2, 0.05),
    (10, '2026-04-04', 5, 3, 5, 0.00),
    (11, '2026-04-10', 1, 2, 2, 0.00),
    (12, '2026-04-16', 4, 5, 12, 0.10),
    (13, '2026-05-05', 6, 1, 7, 0.05),
    (14, '2026-05-13', 2, 4, 3, 0.00),
    (15, '2026-05-25', 3, 5, 6, 0.07),
    (16, '2026-06-02', 5, 1, 4, 0.00),
    (17, '2026-06-11', 4, 2, 3, 0.08),
    (18, '2026-06-17', 6, 3, 9, 0.10),
]

cursor.executemany('INSERT INTO klienci VALUES (?, ?, ?, ?);', klienci)
cursor.executemany('INSERT INTO produkty VALUES (?, ?, ?, ?);', produkty)
cursor.executemany('INSERT INTO sprzedaz VALUES (?, ?, ?, ?, ?, ?);', sprzedaz)

conn.commit()
print('Dane zostały wstawione do SQLite.')


## 5. Odczyt tabel SQLite do ramek pandas

Funkcja `pd.read_sql_query()` pozwala wykonać zapytanie SQL i od razu otrzymać wynik jako `DataFrame`.


In [ ]:
df_klienci = pd.read_sql_query('SELECT * FROM klienci;', conn)
df_produkty = pd.read_sql_query('SELECT * FROM produkty;', conn)
df_sprzedaz = pd.read_sql_query('SELECT * FROM sprzedaz;', conn)

display(df_klienci)
display(df_produkty)
display(df_sprzedaz.head())


## 6. Zapytanie SQL z JOIN i obliczeniem wartości sprzedaży

Poniżej łączymy dane z trzech tabel. Obliczamy wartość pozycji sprzedażowej według wzoru:

`wartosc = ilosc * cena * (1 - rabat)`


In [ ]:
query = '''
SELECT
    s.transakcja_id,
    s.data,
    k.imie,
    k.miasto,
    k.segment,
    p.nazwa AS produkt,
    p.kategoria,
    s.ilosc,
    p.cena,
    s.rabat,
    ROUND(s.ilosc * p.cena * (1 - s.rabat), 2) AS wartosc
FROM sprzedaz s
JOIN klienci k ON s.klient_id = k.klient_id
JOIN produkty p ON s.produkt_id = p.produkt_id
ORDER BY s.data;
'''

df = pd.read_sql_query(query, conn)
df['data'] = pd.to_datetime(df['data'])
df.head(10)


## 7. Analiza w pandas: sprzedaż według miesiąca

Teraz opuszczamy czysty SQL i pracujemy w stylu pandas.


In [ ]:
df['miesiac'] = df['data'].dt.to_period('M').astype(str)

sprzedaz_miesiac = (
    df.groupby('miesiac', as_index=False)
      .agg(
          wartosc_sprzedazy=('wartosc', 'sum'),
          liczba_transakcji=('transakcja_id', 'count'),
          sprzedane_sztuki=('ilosc', 'sum')
      )
      .sort_values('miesiac')
)

sprzedaz_miesiac


## 8. Analiza w pandas: sprzedaż według kategorii i segmentu klienta

To typowy etap analityczny: agregacja, sortowanie, interpretacja.


In [ ]:
sprzedaz_kategoria_segment = (
    df.groupby(['kategoria', 'segment'], as_index=False)
      .agg(
          wartosc_sprzedazy=('wartosc', 'sum'),
          sredni_rabat=('rabat', 'mean'),
          sprzedane_sztuki=('ilosc', 'sum')
      )
      .sort_values('wartosc_sprzedazy', ascending=False)
)

sprzedaz_kategoria_segment['sredni_rabat'] = sprzedaz_kategoria_segment['sredni_rabat'].round(3)
sprzedaz_kategoria_segment


## 9. Parametryzowane zapytanie SQL

W praktyce warto unikać ręcznego sklejania zapytań SQL ze zmiennymi.  
Poniżej filtrujemy sprzedaż dla wybranego miasta za pomocą parametru `?`.


In [ ]:
wybrane_miasto = 'Warszawa'

query_miasto = '''
SELECT
    s.data,
    k.imie,
    k.miasto,
    p.nazwa AS produkt,
    s.ilosc,
    ROUND(s.ilosc * p.cena * (1 - s.rabat), 2) AS wartosc
FROM sprzedaz s
JOIN klienci k ON s.klient_id = k.klient_id
JOIN produkty p ON s.produkt_id = p.produkt_id
WHERE k.miasto = ?
ORDER BY s.data;
'''

df_miasto = pd.read_sql_query(query_miasto, conn, params=(wybrane_miasto,))
df_miasto


## 10. Zapis wyników pandas z powrotem do SQLite

Ramkę pandas można łatwo zapisać do nowej tabeli w bazie SQLite metodą `to_sql()`.


In [ ]:
sprzedaz_miesiac.to_sql('raport_sprzedaz_miesiac', conn, if_exists='replace', index=False)
sprzedaz_kategoria_segment.to_sql('raport_kategoria_segment', conn, if_exists='replace', index=False)

print('Raporty zapisane do tabel SQLite:')
print('- raport_sprzedaz_miesiac')
print('- raport_kategoria_segment')


## 11. Sprawdzenie, jakie tabele istnieją w bazie

To prosty sposób na kontrolę efektu pracy notebooka.


In [ ]:
tabele = pd.read_sql_query("""
SELECT name AS tabela
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

tabele


## 12. Wykres z danych pobranych z SQLite

Pobieramy z bazy tabelę raportową i budujemy prosty wykres sprzedaży miesięcznej.


In [ ]:
df_wykres = pd.read_sql_query('SELECT * FROM raport_sprzedaz_miesiac ORDER BY miesiac;', conn)

ax = df_wykres.plot(
    x='miesiac',
    y='wartosc_sprzedazy',
    kind='bar',
    legend=False,
    figsize=(9, 5),
    title='Wartość sprzedaży według miesiąca'
)

ax.set_xlabel('Miesiąc')
ax.set_ylabel('Wartość sprzedaży')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 13. Eksport raportu do CSV

Na końcu zapisujemy jeden z wyników do pliku CSV. To dobry moment, żeby pokazać uczestnikom pełny mini-pipeline:

`SQLite → pandas → analiza → SQLite → CSV`


In [ ]:
csv_path = Path('raport_sprzedaz_miesiac.csv')
sprzedaz_miesiac.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f'Zapisano plik CSV: {csv_path.resolve()}')


## 14. Zamknięcie połączenia

W małych notebookach często zostawia się połączenie otwarte do końca pracy.  
Dla dobrego stylu pokażmy jednak jawne zamknięcie połączenia.


In [ ]:
conn.close()
print('Połączenie z bazą SQLite zostało zamknięte.')


## Co można pokazać kursantom dalej?

Propozycje rozszerzeń:

1. dodać nową tabelę `dostawcy`,
2. dodać kolumnę `wojewodztwo` do klientów,
3. zrobić ranking TOP 3 produktów według sprzedaży,
4. zbudować prosty dashboard w notebooku,
5. porównać wyniki agregacji wykonanej w SQL i w pandas,
6. zapisać raporty do pliku Excel z wieloma arkuszami.
